In [1]:
# ============================================================
# SMB Churn Model
# Input : gold_smb_account_snapshot
# Output: gold_smb_scored
#         gold_smb_feature_importance
#
# Goal:
# Predict churn risk using account engagement, product breadth,
# support distress, growth trend, and firmographic features.
# ============================================================

from pyspark.sql import functions as F
from pyspark.ml import Pipeline
from pyspark.ml.feature import VectorAssembler, StringIndexer
from pyspark.ml.classification import (
    LogisticRegression,
    RandomForestClassifier,
    GBTClassifier
)
from pyspark.ml.evaluation import (
    BinaryClassificationEvaluator,
    MulticlassClassificationEvaluator
)
from pyspark.ml.functions import vector_to_array

import pandas as pd

GOLD_SNAP    = "gold_smb_account_snapshot"
SCORED_TABLE = "gold_smb_scored"
FI_TABLE     = "gold_smb_feature_importance"

df = spark.table(GOLD_SNAP)

print(f"Gold snapshot rows : {df.count():,}")
print(f"Columns            : {len(df.columns)}")
print(f"Distinct accounts  : {df.select('account_id').distinct().count():,}")

display(df.limit(5))

StatementMeta(, 084b6c44-0453-475a-8b4b-8be0f131dff8, 3, Finished, Available, Finished, False)

Gold snapshot rows : 5,000
Columns            : 41
Distinct accounts  : 5,000


SynapseWidget(Synapse.DataFrame, f5a86432-5ce6-42c1-affb-169182537233)

In [2]:
# Quick validation


print("Churn distribution:")
display(
    df.groupBy("churned")
      .count()
      .orderBy("churned")
)

required_cols = [
    "avg_active_user_rate_6mo",
    "active_user_rate",
    "product_breadth_score",
    "months_without_growth",
    "support_spike_flag",
    "azure_active_flag",
    "teams_adoption_pct",
    "employee_count",
    "seat_growth_mom",
    "usage_momentum",
    "revenue_tier",
    "trend_direction",
    "industry",
    "region",
    "employee_band"
]

missing = [c for c in required_cols if c not in df.columns]

if missing:
    print(f"MISSING COLUMNS - fix Gold before ML: {missing}")
else:
    print("All required ML columns present ✓")

null_summary = df.select([
    F.sum(F.when(F.col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in required_cols
])

print("Null summary for ML columns:")
display(null_summary)

StatementMeta(, 084b6c44-0453-475a-8b4b-8be0f131dff8, 4, Finished, Available, Finished, False)

Churn distribution:


SynapseWidget(Synapse.DataFrame, 39190af9-bc04-4ec4-98ac-69246168df4d)

All required ML columns present ✓
Null summary for ML columns:


SynapseWidget(Synapse.DataFrame, 66b02c85-e480-43f6-8f5c-b193ae61c394)

In [3]:
# Feature selection + null handling

# IMPORTANT:
# We are intentionally NOT using:
# - recommended_action
# - retention_priority_score
# - retention_risk_flag
# - expansion_ready_flag
# - revenue_at_risk
# - health_label
# - health_score
#
# Reason:
# These are downstream business outputs or composite scores.
# Excluding them reduces leakage risk and keeps the model cleaner.

CONTINUOUS_FEATURES = [
    "avg_active_user_rate_6mo",
    "active_user_rate",
    "product_breadth_score",
    "months_without_growth",
    "support_spike_flag",
    "azure_active_flag",
    "teams_adoption_pct",
    "employee_count",
    "seat_growth_mom",
    "usage_momentum"
]

CATEGORICAL_FEATURES = [
    "revenue_tier",
    "trend_direction",
    "industry",
    "region",
    "employee_band"
]

LABEL_COL   = "churned"
FEATURE_COL = "features"

df_ml = df

# Fill numeric nulls with 0.0
for col in CONTINUOUS_FEATURES:
    df_ml = df_ml.withColumn(col, F.coalesce(F.col(col).cast("double"), F.lit(0.0)))

# Fill categorical nulls with "Unknown"
for col in CATEGORICAL_FEATURES:
    df_ml = df_ml.withColumn(col, F.coalesce(F.col(col), F.lit("Unknown")))

# Cast label
df_ml = df_ml.withColumn(LABEL_COL, F.col(LABEL_COL).cast("double"))

print(f"Rows available for ML: {df_ml.count():,}")
print(f"Churn rate: {df_ml.filter(F.col(LABEL_COL) == 1).count() / df_ml.count():.1%}")

StatementMeta(, 084b6c44-0453-475a-8b4b-8be0f131dff8, 5, Finished, Available, Finished, False)

Rows available for ML: 5,000
Churn rate: 11.9%


In [4]:
# Build preprocessing objects

indexers = [
    StringIndexer(
        inputCol=cat,
        outputCol=f"{cat}_idx",
        handleInvalid="keep"
    )
    for cat in CATEGORICAL_FEATURES
]

assembler = VectorAssembler(
    inputCols=CONTINUOUS_FEATURES + [f"{c}_idx" for c in CATEGORICAL_FEATURES],
    outputCol=FEATURE_COL,
    handleInvalid="keep"
)

print("Preprocessing objects created successfully.")

StatementMeta(, 084b6c44-0453-475a-8b4b-8be0f131dff8, 6, Finished, Available, Finished, False)

Preprocessing objects created successfully.


In [5]:
# Train/test split

# Note: this is a random split, not a stratified split.

train_df, test_df = df_ml.randomSplit([0.8, 0.2], seed=42)

train_total = train_df.count()
test_total  = test_df.count()

train_churn = train_df.filter(F.col(LABEL_COL) == 1).count()
test_churn  = test_df.filter(F.col(LABEL_COL) == 1).count()

print(f"Training rows : {train_total:,}")
print(f"Test rows     : {test_total:,}")
print(f"Train churn % : {train_churn / train_total:.1%}")
print(f"Test churn %  : {test_churn / test_total:.1%}")

StatementMeta(, 084b6c44-0453-475a-8b4b-8be0f131dff8, 7, Finished, Available, Finished, False)

Training rows : 4,040
Test rows     : 960
Train churn % : 12.0%
Test churn %  : 11.9%


In [6]:
# Model comparison - Logistic Regression vs RF vs GBT

auc_eval = BinaryClassificationEvaluator(
    labelCol=LABEL_COL,
    rawPredictionCol="rawPrediction",
    metricName="areaUnderROC"
)

acc_eval = MulticlassClassificationEvaluator(
    labelCol=LABEL_COL,
    predictionCol="prediction",
    metricName="accuracy"
)

f1_eval = MulticlassClassificationEvaluator(
    labelCol=LABEL_COL,
    predictionCol="prediction",
    metricName="f1"
)

models_to_compare = {
    "Logistic Regression": LogisticRegression(
        labelCol=LABEL_COL,
        featuresCol=FEATURE_COL,
        maxIter=100,
        regParam=0.01
    ),
    "Random Forest": RandomForestClassifier(
        labelCol=LABEL_COL,
        featuresCol=FEATURE_COL,
        predictionCol="prediction",
        probabilityCol="probability",
        rawPredictionCol="rawPrediction",
        numTrees=120,
        maxDepth=5,
        seed=42
    ),
    "Gradient Boosted Trees": GBTClassifier(
        labelCol=LABEL_COL,
        featuresCol=FEATURE_COL,
        predictionCol="prediction",
        maxIter=50,
        maxDepth=5,
        seed=42
    )
}

comparison_results = []

for model_name, classifier in models_to_compare.items():
    print(f"Training {model_name}...")

    comparison_pipeline = Pipeline(stages=indexers + [assembler, classifier])
    trained_model = comparison_pipeline.fit(train_df)
    preds = trained_model.transform(test_df)

    auc = auc_eval.evaluate(preds)
    acc = acc_eval.evaluate(preds)
    f1  = f1_eval.evaluate(preds)

    comparison_results.append((model_name, round(auc, 4), round(acc, 4), round(f1, 4)))

comparison_df = spark.createDataFrame(
    comparison_results,
    ["model_name", "auc_roc", "accuracy", "f1_score"]
).orderBy(F.desc("auc_roc"))

print("=" * 60)
print("MODEL COMPARISON SUMMARY")
print("=" * 60)
display(comparison_df)

best_model_name = comparison_df.first()["model_name"]
print(f"Best model by AUC-ROC: {best_model_name}")

StatementMeta(, 084b6c44-0453-475a-8b4b-8be0f131dff8, 8, Finished, Available, Finished, False)

Training Logistic Regression...


Training Random Forest...


Training Gradient Boosted Trees...


MODEL COMPARISON SUMMARY


SynapseWidget(Synapse.DataFrame, 9dff17ce-76d8-4761-89f0-66dea3aaebab)

Best model by AUC-ROC: Random Forest


Why AUC matters more here?
- The Churn rate is only about 12%, so this is an imbalanced classification problem.

In imbalanced problems:
- Accuracy can be misleading
- AUC-ROC is usually a stronger selection metric. AUC measures ranking/discrimination quality across thresholds.
- F1 is also useful, but AUC is better for overall ranking/discrimination

That is why choosing Random Forest is still defensible.

In [7]:
# Model selection rationale

comparison_pd = comparison_df.toPandas()

rf_row = comparison_pd[comparison_pd["model_name"] == "Random Forest"].iloc[0]
gbt_row = comparison_pd[comparison_pd["model_name"] == "Gradient Boosted Trees"].iloc[0]
best_row = comparison_pd.iloc[0]

print("FINAL MODEL SELECTION LOGIC")
print("=" * 60)
print(f"Top model by AUC-ROC: {best_row['model_name']}")
print(f"RF  -> AUC: {rf_row['auc_roc']:.4f}, Accuracy: {rf_row['accuracy']:.4f}, F1: {rf_row['f1_score']:.4f}")
print(f"GBT -> AUC: {gbt_row['auc_roc']:.4f}, Accuracy: {gbt_row['accuracy']:.4f}, F1: {gbt_row['f1_score']:.4f}")

# Decision rule:
# - Prioritize AUC-ROC as primary metric because churn is imbalanced
# - If GBT beats RF by more than 0.02 AUC, switch to GBT
# - Otherwise keep RF for stronger explainability and cleaner feature importance

selected_model_name = "Random Forest"

if gbt_row["auc_roc"] - rf_row["auc_roc"] > 0.02:
    selected_model_name = "Gradient Boosted Trees"

print(f"\nSelected final model: {selected_model_name}")

if selected_model_name == "Random Forest":
    print("""
Why Random Forest was finalized:
1. It achieved the highest AUC-ROC in the benchmark
2. Although GBT was slightly higher on Accuracy and F1, the AUC gap favored RF
3. AUC-ROC was prioritized because churn is an imbalanced classification problem
4. Random Forest is easier to explain to business stakeholders
5. Feature importance is more straightforward for dashboard and interview storytelling
""")
else:
    print("""
Why Gradient Boosted Trees was finalized:
1. It materially outperformed Random Forest on AUC-ROC
2. Better nonlinear discrimination on churn behavior justified the switch
3. Performance gain outweighed the loss in interpretability
""")

StatementMeta(, 084b6c44-0453-475a-8b4b-8be0f131dff8, 9, Finished, Available, Finished, False)

FINAL MODEL SELECTION LOGIC
Top model by AUC-ROC: Random Forest
RF  -> AUC: 0.9989, Accuracy: 0.9896, F1: 0.9896
GBT -> AUC: 0.9975, Accuracy: 0.9927, F1: 0.9927

Selected final model: Random Forest

Why Random Forest was finalized:
1. It achieved the highest AUC-ROC in the benchmark
2. Although GBT was slightly higher on Accuracy and F1, the AUC gap favored RF
3. AUC-ROC was prioritized because churn is an imbalanced classification problem
4. Random Forest is easier to explain to business stakeholders
5. Feature importance is more straightforward for dashboard and interview storytelling



In [8]:
# Train FINAL selected model (AFTER comparison)

print(f"Training FINAL model: {selected_model_name}")

if selected_model_name == "Random Forest":
    final_model = RandomForestClassifier(
        labelCol=LABEL_COL,
        featuresCol=FEATURE_COL,
        predictionCol="prediction",
        probabilityCol="probability",
        rawPredictionCol="rawPrediction",
        numTrees=120,
        maxDepth=5,
        seed=42
    )

elif selected_model_name == "Gradient Boosted Trees":
    final_model = GBTClassifier(
        labelCol=LABEL_COL,
        featuresCol=FEATURE_COL,
        predictionCol="prediction",
        maxIter=50,
        maxDepth=5,
        seed=42
    )

else:  # Logistic Regression fallback
    final_model = LogisticRegression(
        labelCol=LABEL_COL,
        featuresCol=FEATURE_COL,
        maxIter=100,
        regParam=0.01
    )

final_pipeline = Pipeline(stages=indexers + [assembler, final_model])
model = final_pipeline.fit(train_df)

print("Final model training complete.")

StatementMeta(, 084b6c44-0453-475a-8b4b-8be0f131dff8, 10, Finished, Available, Finished, False)

Training FINAL model: Random Forest


Final model training complete.


In [9]:
# Evaluate final SELECTED model

predictions = model.transform(test_df)

predictions = predictions.withColumn(
    "churn_probability",
    F.round(vector_to_array("probability")[1], 4)
)

auc = auc_eval.evaluate(predictions)
acc = acc_eval.evaluate(predictions)
f1  = f1_eval.evaluate(predictions)

print("=" * 50)
print("FINAL MODEL EVALUATION RESULTS")
print("=" * 50)
print(f"AUC-ROC  : {auc:.4f}")
print(f"Accuracy : {acc:.4f}")
print(f"F1 Score : {f1:.4f}")
print("=" * 50)

print("Confusion Matrix:")
display(
    predictions.groupBy(LABEL_COL, "prediction")
               .count()
               .orderBy(LABEL_COL, "prediction")
)

StatementMeta(, 084b6c44-0453-475a-8b4b-8be0f131dff8, 11, Finished, Available, Finished, False)

FINAL MODEL EVALUATION RESULTS
AUC-ROC  : 0.9980
Accuracy : 0.9896
F1 Score : 0.9896
Confusion Matrix:


SynapseWidget(Synapse.DataFrame, 2c6a03f6-403e-4854-a19e-173c3ca8c6cc)

In [10]:
# Class distribution check


df.groupBy("churned").count().show()

total = df.count()
churned = df.filter("churned = 1").count()
print(f"Churn rate: {churned/total:.2%}")

StatementMeta(, 084b6c44-0453-475a-8b4b-8be0f131dff8, 12, Finished, Available, Finished, False)

+-------+-----+
|churned|count|
+-------+-----+
|      1|  597|
|      0| 4403|
+-------+-----+

Churn rate: 11.94%


In [11]:
# Threshold tuning

from pyspark.ml.functions import vector_to_array
import pyspark.sql.functions as F

# Convert probability vector → churn probability
pred_df = predictions.withColumn(
    "churn_prob",
    vector_to_array("probability")[1]
)

# Test multiple thresholds
thresholds = [0.3, 0.4, 0.5, 0.6, 0.7]

results = []

for t in thresholds:
    temp = pred_df.withColumn(
        "pred_label",
        (F.col("churn_prob") > t).cast("int")
    )
    
    tp = temp.filter("pred_label = 1 AND churned = 1").count()
    fp = temp.filter("pred_label = 1 AND churned = 0").count()
    fn = temp.filter("pred_label = 0 AND churned = 1").count()
    
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    
    results.append((t, precision, recall))

spark.createDataFrame(results, ["threshold", "precision", "recall"]).show()

StatementMeta(, 084b6c44-0453-475a-8b4b-8be0f131dff8, 13, Finished, Available, Finished, False)

+---------+------------------+------------------+
|threshold|         precision|            recall|
+---------+------------------+------------------+
|      0.3|0.8740157480314961|0.9736842105263158|
|      0.4|             0.925|0.9736842105263158|
|      0.5| 0.956140350877193| 0.956140350877193|
|      0.6|0.9557522123893806|0.9473684210526315|
|      0.7| 0.970873786407767|0.8771929824561403|
+---------+------------------+------------------+



I tested multiple probability thresholds instead of accepting the default 0.5 cutoff. The tradeoff is business-driven: a lower threshold catches more at-risk accounts but generates more false alarms for the sales team. 

 For this project I kept 0.5 as the production threshold because precision and recall are balanced at 95.6% each, meaning the sales team's call list is accurate and we're not missing significant revenue.

In [12]:
# ROC (AUC) check

roc = auc_eval.evaluate(predictions)
print(f"AUC (ROC): {roc:.4f}")

StatementMeta(, 084b6c44-0453-475a-8b4b-8be0f131dff8, 14, Finished, Available, Finished, False)

AUC (ROC): 0.9980


In [13]:
# Business segmentation

from pyspark.ml.functions import vector_to_array
import pyspark.sql.functions as F

scored_df = predictions.withColumn(
    "churn_probability",
    vector_to_array("probability")[1]
)

scored_df = scored_df.withColumn(
    "churn_risk_label",
    F.when(F.col("churn_probability") < 0.3, "Low")
     .when(F.col("churn_probability") < 0.6, "Medium")
     .when(F.col("churn_probability") < 0.8, "High")
     .otherwise("Critical")
)

scored_df.select("churn_probability", "churn_risk_label").show(10)

StatementMeta(, 084b6c44-0453-475a-8b4b-8be0f131dff8, 15, Finished, Available, Finished, False)

+--------------------+----------------+
|   churn_probability|churn_risk_label|
+--------------------+----------------+
| 0.05459149234986492|             Low|
|0.003185934546800...|             Low|
|0.003255565636683...|             Low|
|  0.8012534220871907|        Critical|
|0.005389505310833013|             Low|
|0.004130739646011402|             Low|
|   0.956903319671988|        Critical|
|0.006671397800019013|             Low|
|0.003146008554412...|             Low|
|0.003178368650647...|             Low|
+--------------------+----------------+
only showing top 10 rows



In [14]:
# Feature importance

rf_model = model.stages[-1]

all_feature_names = CONTINUOUS_FEATURES + [f"{c}_idx" for c in CATEGORICAL_FEATURES]
importances = rf_model.featureImportances.toArray()

fi_list = sorted(
    zip(all_feature_names, importances),
    key=lambda x: x[1],
    reverse=True
)

print("TOP FEATURE IMPORTANCE RANKING")
print("=" * 60)
for rank, (feat, imp) in enumerate(fi_list[:15], 1):
    print(f"{rank:>2}. {feat:<30} {imp:.6f}")

fi_pdf = pd.DataFrame(fi_list, columns=["feature_name", "importance"])
fi_spark = spark.createDataFrame(fi_pdf)

fi_spark = fi_spark.withColumn("importance", F.round(F.col("importance"), 6))
fi_spark.write.mode("overwrite").saveAsTable(FI_TABLE)

print(f"{FI_TABLE} saved successfully.")
display(
    spark.table(FI_TABLE).orderBy(F.desc("importance")).limit(15)
)

StatementMeta(, 084b6c44-0453-475a-8b4b-8be0f131dff8, 16, Finished, Available, Finished, False)

TOP FEATURE IMPORTANCE RANKING
 1. support_spike_flag             0.394129
 2. active_user_rate               0.270060
 3. usage_momentum                 0.101494
 4. trend_direction_idx            0.099863
 5. teams_adoption_pct             0.066385
 6. avg_active_user_rate_6mo       0.034015
 7. seat_growth_mom                0.008162
 8. employee_count                 0.007596
 9. product_breadth_score          0.004185
10. employee_band_idx              0.003731
11. months_without_growth          0.003686
12. industry_idx                   0.003075
13. region_idx                     0.001891
14. revenue_tier_idx               0.001190
15. azure_active_flag              0.000539
gold_smb_feature_importance saved successfully.


SynapseWidget(Synapse.DataFrame, c90571c9-552e-4cb4-9f1d-3ff74721d94e)

In [15]:
# Score all accounts


all_scored = model.transform(df_ml)

all_scored = all_scored.withColumn(
    "churn_probability",
    F.round(vector_to_array("probability")[1], 4)
)

all_scored = all_scored.withColumn(
    "churn_risk_label",
    F.when(F.col("churn_probability") >= 0.80, "Critical")
     .when(F.col("churn_probability") >= 0.60, "High")
     .when(F.col("churn_probability") >= 0.35, "Medium")
     .otherwise("Low")
)

print("Churn risk distribution:")
display(
    all_scored.groupBy("churn_risk_label")
              .count()
              .orderBy(F.desc("count"))
)

StatementMeta(, 084b6c44-0453-475a-8b4b-8be0f131dff8, 17, Finished, Available, Finished, False)

Churn risk distribution:


SynapseWidget(Synapse.DataFrame, 5355c1d1-e41f-4d1f-89f8-9e16aaac3abd)

In [16]:
# ============================================================
# Decision Layer = Probability-Aware Actions + Explainability
# ============================================================
# This cell runs AFTER ML scoring so recommended_action is driven by churn_probability directly, not by the rule-based health label from Gold.
#
# Three new columns:
#   recommended_action  - overrides Gold version with prob-aware logic
#   decision_reason     - plain English explanation for the action
#   key_driver          - the single strongest behavioral signal
# ============================================================

all_scored = all_scored.withColumn(
    "recommended_action",
    F.when(
        (F.col("churn_probability") >= 0.80) &
        (F.col("product_breadth_score") == 1),
        "Critical Retention Call — Single Product Account"
    ).when(
        (F.col("churn_probability") >= 0.80) &
        (F.col("revenue_tier") == "High"),
        "Critical Retention Call — High Value Account"
    ).when(
        F.col("churn_probability") >= 0.80,
        "Urgent Account Health Review"
    ).when(
        (F.col("churn_probability") >= 0.60) &
        (F.col("azure_active_flag") == 0),
        "Outreach: Azure Trial Campaign"
    ).when(
        (F.col("churn_probability") >= 0.60) &
        (F.col("dynamics_users") == 0),
        "Outreach: Dynamics 365 Introduction"
    ).when(
        (F.col("churn_probability") < 0.35) &
        (F.col("active_user_rate") >= 0.75) &
        (F.col("product_breadth_score") >= 2),
        "Upsell: Copilot License Recommendation"
    ).when(
        F.col("churn_probability") < 0.30,
        "Nurture: Community Program"
    ).otherwise(
        "Monitor: Standard Check-in"
    )
)

# ---- Decision reason - why was this action assigned? ----
all_scored = all_scored.withColumn(
    "decision_reason",
    F.when(
        F.col("churn_probability") >= 0.80,
        "Churn probability exceeds critical threshold (80%)"
    ).when(
        (F.col("churn_probability") >= 0.60) &
        (F.col("support_spike_flag") == 1),
        "Elevated churn risk combined with support ticket spike"
    ).when(
        (F.col("churn_probability") >= 0.60) &
        (F.col("trend_direction") == "Declining"),
        "Elevated churn risk with declining usage trend"
    ).when(
        (F.col("churn_probability") >= 0.60) &
        (F.col("azure_active_flag") == 0),
        "Elevated churn risk — no Azure ecosystem protection"
    ).when(
        (F.col("churn_probability") < 0.35) &
        (F.col("active_user_rate") >= 0.75),
        "Strong engagement and multi-product adoption — expansion ready"
    ).when(
        F.col("churn_probability") < 0.30,
        "Low churn risk — healthy engagement signals"
    ).otherwise(
        "Moderate risk — standard monitoring cadence"
    )
)

StatementMeta(, 084b6c44-0453-475a-8b4b-8be0f131dff8, 18, Finished, Available, Finished, False)

In [19]:
# ---- Key driver - the single strongest behavioral signal ----
# Based on feature importance ranking:
# support_spike_flag (39.4%) > active_user_rate (27.0%) >
# usage_momentum (10.1%) > trend_direction (10.0%) > teams_adoption_pct (6.6%)
all_scored = all_scored.withColumn(
    "key_driver",
    F.when(
        F.col("support_spike_flag") == 1,
        "Support Spike — elevated ticket volume detected"
    ).when(
        F.col("active_user_rate") < 0.40,
        "Low Active User Rate — seat utilization critically low"
    ).when(
        F.col("usage_momentum") < -0.10,
        "Rapid Usage Decline — engagement falling sharply"
    ).when(
        F.col("trend_direction") == "Declining",
        "Declining Trend — consistent engagement deterioration"
    ).when(
        F.col("teams_adoption_pct") < 0.30,
        "Low Teams Adoption — collaboration depth weak"
    ).when(
        F.col("months_without_growth") >= 4,
        "Prolonged Stagnation — no seat growth for 4+ months"
    ).otherwise(
        "Stable Behavioral Profile"
    )
)

# Verify the three new columns
print("New columns added: recommended_action, decision_reason, key_driver")
print("\nAction distribution (probability-aware):")
display(
    all_scored.groupBy("recommended_action")
    .count()
    .orderBy(F.desc("count"))
)

print("\nSample - Critical accounts with full context:")
display(
    all_scored.filter(F.col("churn_probability") >= 0.80)
    .select(
        "company_name", "revenue_tier", "churn_probability",
        "recommended_action", "decision_reason", "key_driver"
    )
    .orderBy(F.desc("churn_probability"))
    .limit(10)
)

StatementMeta(, 084b6c44-0453-475a-8b4b-8be0f131dff8, 21, Finished, Available, Finished, False)

New columns added: recommended_action, decision_reason, key_driver

Action distribution (probability-aware):


SynapseWidget(Synapse.DataFrame, db9f7da7-5819-49b2-8b43-b55ff6a2f243)


Sample - Critical accounts with full context:


SynapseWidget(Synapse.DataFrame, 648d5cd9-cbe1-4d74-832f-5ff918bc9ff2)

In [20]:
# Save gold_smb_scored - Final scored table with decision layer

base_cols = [c for c in df.columns if c not in
             ["recommended_action"]]  # exclude old version from Gold

gold_scored = all_scored.select(
    *base_cols,
    "churn_probability",
    "churn_risk_label",
    "recommended_action",   # probability-aware version
    "decision_reason",      # new
    "key_driver"            # new
)

gold_scored.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(SCORED_TABLE)

saved = spark.table(SCORED_TABLE)
print(f"{SCORED_TABLE} saved successfully.")
print(f"Rows    : {saved.count():,}")
print(f"Columns : {len(saved.columns)}")

StatementMeta(, 084b6c44-0453-475a-8b4b-8be0f131dff8, 22, Finished, Available, Finished, False)

gold_smb_scored saved successfully.
Rows    : 5,000
Columns : 45


In [21]:
# Final validation

scored = spark.table(SCORED_TABLE)

print("=" * 60)
print("GOLD_SMB_SCORED - FINAL VALIDATION")
print("=" * 60)

print(f"Total accounts scored: {scored.count():,}")
print(f"Total columns        : {len(scored.columns)}")

bad_prob = scored.filter(
    (F.col("churn_probability") < 0) | (F.col("churn_probability") > 1)
).count()

print(f"churn_probability outside [0,1]: {bad_prob} {'✓' if bad_prob == 0 else '✗'}")

print("\nChurn risk label distribution:")
display(
    scored.groupBy("churn_risk_label")
          .count()
          .orderBy(F.desc("count"))
)

print("\nRecommended action distribution:")
display(
    scored.groupBy("recommended_action")
          .count()
          .orderBy(F.desc("count"))
)

print("\nDecision layer null check:")
display(
    scored.select(
        F.sum(F.when(F.col("recommended_action").isNull(), 1).otherwise(0)).alias("null_recommended_action"),
        F.sum(F.when(F.col("decision_reason").isNull(), 1).otherwise(0)).alias("null_decision_reason"),
        F.sum(F.when(F.col("key_driver").isNull(), 1).otherwise(0)).alias("null_key_driver")
    )
)

print("\nDecision layer preview:")
display(
    scored.select(
        "company_name",
        "churn_probability",
        "churn_risk_label",
        "recommended_action",
        "decision_reason",
        "key_driver"
    ).limit(10)
)

print("\nTop 10 highest-risk accounts:")
display(
    scored.orderBy(F.desc("churn_probability"))
          .select(
              "company_name",
              "industry",
              "region",
              "revenue_tier",
              "health_score",
              "health_label",
              "estimated_clv_12mo",
              "churn_probability",
              "churn_risk_label",
              "recommended_action",
              "decision_reason",
              "key_driver"
          )
          .limit(10)
)

StatementMeta(, 084b6c44-0453-475a-8b4b-8be0f131dff8, 23, Finished, Available, Finished, False)

GOLD_SMB_SCORED - FINAL VALIDATION
Total accounts scored: 5,000
Total columns        : 45
churn_probability outside [0,1]: 0 ✓

Churn risk label distribution:


SynapseWidget(Synapse.DataFrame, 0772ec61-b89a-4e39-a2fd-2b5cafbe48bf)


Recommended action distribution:


SynapseWidget(Synapse.DataFrame, 4e1d22d8-155d-4fca-861c-079a77a0d4f2)

...

SynapseWidget(Synapse.DataFrame, e52c213b-67c5-435a-b2c1-d2fa29b6b2ef)


Decision layer preview:


SynapseWidget(Synapse.DataFrame, 15c283dd-8760-4a8b-bb3d-6c304b994ca2)


Top 10 highest-risk accounts:


SynapseWidget(Synapse.DataFrame, c9dbe829-f943-4842-b3da-6ea70b22a377)

In [24]:
# Modeling rationale


print("""
I benchmarked three models before finalizing the churn model:
1. Logistic Regression as a baseline
2. Random Forest as the primary candidate
3. Gradient Boosted Trees as a challenger

In the benchmark:
- Random Forest achieved the highest AUC-ROC
- Gradient Boosted Trees was slightly higher on Accuracy and F1
- Logistic Regression underperformed relative to both tree-based models

I selected Random Forest as the final model because:
- AUC-ROC was prioritized as the primary metric due to class imbalance (~12% churn rate)
- Random Forest slightly outperformed GBT on AUC
- It provides more interpretable and stable feature importance for business use

I also evaluated class imbalance and tested multiple probability thresholds
to better understand precision-recall tradeoffs instead of relying on a fixed 0.5 cutoff.

I intentionally excluded downstream business outputs such as recommended_action,
retention_priority_score, and revenue_at_risk to reduce leakage and ensure the model
learned from account behavior rather than from decision rules.

After scoring, I added a probability-aware decision layer that translates churn
probabilities into business-facing outputs: recommended_action, decision_reason,
and key_driver. This makes the final output directly usable for prioritization
and intervention.
""")

StatementMeta(, 084b6c44-0453-475a-8b4b-8be0f131dff8, 26, Finished, Available, Finished, False)


I benchmarked three models before finalizing the churn model:
1. Logistic Regression as a baseline
2. Random Forest as the primary candidate
3. Gradient Boosted Trees as a challenger

In the benchmark:
- Random Forest achieved the highest AUC-ROC
- Gradient Boosted Trees was slightly higher on Accuracy and F1
- Logistic Regression underperformed relative to both tree-based models

I selected Random Forest as the final model because:
- AUC-ROC was prioritized as the primary metric due to class imbalance (~12% churn rate)
- Random Forest slightly outperformed GBT on AUC
- It provides more interpretable and stable feature importance for business use

I also evaluated class imbalance and tested multiple probability thresholds
to better understand precision-recall tradeoffs instead of relying on a fixed 0.5 cutoff.

I intentionally excluded downstream business outputs such as recommended_action,
retention_priority_score, and revenue_at_risk to reduce leakage and ensure the model
learned fr